# 18f — E20 guarded runs: the consistent-set arms (PF-2) and the block-split arms (PF-3)

Study plan v0.18 post-freeze sequence. **Step 2 (PF-2):** S0 at SSP585 under the selected shapes — refugia `log1p(1/v)`, transboundary
connectivity `I²`, climate corridors `I²` (Ethan's call; provenance in M4.35 addendum 3) — as `e20_consistent` (weights only, t = 1)
and `e20_consistent_target` (with R2's connectivity target 0.198). **Step 3 (PF-3, M4.36), gated on the step-2 baseline:** the same
shapes under two block splits — (A) five discretionary blocks at 20% with transboundary and corridors separate; (B) corridors removed,
transboundary alone at the connectivity block's share — each on S0 and S2 (connectivity-forward): four more runs. Every arm: anchor at
opt_gap 1e-4, guarded MGA (k = 50, g = 5%, per-block floors on the ARM's blocks) into `runs_v3.1/<arm>/s0_ssp585_theta5/`. Resumable
per arm; ~1 h each (~6 h for all six). Then 18g.

In [1]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))
mpath <- pr_refresh_manifest(PROJ, ANALYSIS)
VERSION <- "v3.1"
MANIFEST_REL <- sprintf("analyses/y2y/spec/manifest_%s.csv", VERSION); FREEZE_REL <- sprintf("analyses/y2y/spec/manifest_%s.sha256", VERSION)
RUNS_REL <- sprintf("analyses/y2y/runs_%s", VERSION); EFG_SUBDIR_EXPECTED <- paste0("iucn_efg_", sub("\\..*$", "", VERSION))
MAN <- read.csv(file.path(PROJ, MANIFEST_REL), stringsAsFactors = FALSE)
dig <- strsplit(readLines(file.path(PROJ, FREEZE_REL))[1], "  ")[[1]][1]
stopifnot(identical(unname(tools::sha256sum(file.path(PROJ, MANIFEST_REL))[[1]]), dig), nrow(MAN) == 12)
E20 <- jsonlite::read_json(file.path(PROJ, sprintf("analyses/y2y/spec/%s/e20_refugia_transform.json", VERSION)))
row <- MAN[MAN$formulation_id == E20$base_formulation, ]; stopifnot(nrow(row) == 1)
ER <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/e_round_v13.json"))
BLOCKS <- lapply(ER$e17_t3$blocks, unlist); FLOOR_G <- as.numeric(E20$floor_g)
ARM_NAMES <- c("e20_consistent", "e20_consistent_target",                       # step 2: shape only (t = 1) and shape + R2 target (t = 0.198)
               "e20_pf3_A_s0", "e20_pf3_A_s2", "e20_pf3_B_s0", "e20_pf3_B_s2")   # step 3: block splits (gated on the step-2 baseline)
# a context with every patched value-shape layer swapped in BEFORE ingest (the 245-realization mechanism), then the identity check:
# ingest sum-normalizes every feature, so the check compares max/mean (scale-invariant) with 18d's record
build_ctx <- function(ARM) {
  ctx <- pr_setup(mpath, PROJ)
  for (nm in names(ARM$layer_patch)) ctx$layers$path[ctx$layers$name == nm] <- ARM$layer_patch[[nm]]
  ctx <- modifyList(ctx, pr_ingest(ctx)); ctx <- modifyList(ctx, pr_planning_units(ctx))
  stopifnot(all(grepl(paste0("/", EFG_SUBDIR_EXPECTED, "/"), ctx$layers$path[ctx$layers$role == "feature_efg"])))
  for (nm in names(ARM$layer_check)) {
    v <- terra::values(ctx$features[[nm]]); r_obs <- max(v, na.rm = TRUE) / mean(v, na.rm = TRUE); r_exp <- as.numeric(ARM$layer_check[[nm]])
    if (!(abs(r_obs / r_exp - 1) < 0.02)) stop(sprintf("%s: the ingested feature is not the test layer (max/mean %.2f vs expected %.2f)", nm, r_obs, r_exp))
    cat(sprintf("   %s: test layer confirmed (max/mean %.2f)\n", nm, r_obs))
  }
  ctx
}
cat(sprintf("E20 PF-2: arms %s | k %d g %.2f floor %.2f\n", paste(ARM_NAMES, collapse = ", "), as.integer(E20$k), as.numeric(E20$band_g), FLOOR_G))


manifest refreshed from config.py (analysis=y2y)
E20 PF-2: arms e20_consistent, e20_consistent_target, e20_pf3_A_s0, e20_pf3_A_s2, e20_pf3_B_s0, e20_pf3_B_s2 | k 50 g 0.05 floor 0.05


In [2]:
# ---- anchor + guarded MGA per arm (mirrors 18's run_guard, without the frozen-anchor assert: these anchors are new) ---------
for (ARM_NAME in ARM_NAMES) {
  ARM <- E20$arms[[ARM_NAME]]; stopifnot(!is.null(ARM))
  OUT_REL <- file.path(RUNS_REL, ARM_NAME, E20$base_formulation); OUT <- file.path(PROJ, OUT_REL); dir.create(OUT, recursive = TRUE, showWarnings = FALSE)
  tif <- file.path(OUT, "mga_guard_g05.tif")
  if (file.exists(tif)) { cat(sprintf("%s: guard tif exists -- skipped (delete the folder to re-run)\n", ARM_NAME)); next }
  if (!is.null(ARM$requires) && !file.exists(file.path(PROJ, ARM$requires))) { cat(sprintf("%s: step-2 baseline missing (%s) -- not run (spec: do not proceed to step 3 first)\n", ARM_NAME, ARM$requires)); next }
  BLOCKS_ARM <- if (!is.null(ARM$blocks)) lapply(ARM$blocks, unlist) else BLOCKS      # PF-3 arms carry their own block structure (floors follow it)
  cat(sprintf("== %s: %s\n", ARM_NAME, ARM$label))
  ctx <- build_ctx(ARM)
  w <- unlist(ARM$weights); t <- unlist(if (!is.null(ARM$targets)) ARM$targets else E20$targets)
  cat(sprintf("   transboundary target %s | refugia weight %.4f (registered %.4f) | k %d g %.2f floor %.2f -> %s\n",
              if (!("transboundary_connectivity" %in% names(t))) "1 (registered)" else sprintf("%.3f (R2)", t[["transboundary_connectivity"]]), w[["climate_type_macrorefugia"]],
              unlist(E20$weights_registered)[["climate_type_macrorefugia"]], as.integer(E20$k), as.numeric(E20$band_g), FLOOR_G, OUT_REL))
  actx <- pr_override(ctx, targets = as.list(t), feature_weight_multipliers = as.list(w),
                      results_dir = OUT_REL, results_subdir = "guard_build",
                      solver = "gurobi", decision_type = "binary", opt_gap = as.numeric(E20$opt_gap), portfolio_n = 1)
  actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  cm <- mga_compile(actx)
  t0 <- proc.time()[["elapsed"]]
  anchor <- mga_anchor(cm, opt_gap = as.numeric(E20$opt_gap))
  write_layers <- function(M, path, prefix) {
    layers <- lapply(seq_len(nrow(M)), function(i) {
      r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r)); v[cm$pu_index] <- as.integer(M[i, ]); terra::values(r) <- v; r })
    s <- terra::rast(layers); names(s) <- sprintf("%s_%02d", prefix, seq_len(nrow(M)))
    terra::writeRaster(s, path, overwrite = TRUE, datatype = "INT1U", NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  }
  write_layers(matrix(anchor$x, nrow = 1), file.path(OUT, "anchor.tif"), "anchor")
  jsonlite::write_json(list(formulation_id = E20$base_formulation, variant = ARM$label, layer_patch = ARM$layer_patch,
                            anchor_objective = anchor$z, anchor_bound = anchor$bound, anchor_gap = anchor$gap, anchor_runtime_s = anchor$runtime,
                            n_selected = sum(anchor$x), weights = as.list(w), targets = as.list(t),
                            created_utc = format(Sys.time(), tz = "UTC")),
                       file.path(OUT, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
  floors <- list(ctx = actx, blocks = BLOCKS_ARM, g = FLOOR_G)
  gen <- mga_generate(cm, anchor, g = as.numeric(E20$band_g), k = as.integer(E20$k), floors = floors)
  write_layers(gen$members, tif, "guard")
  write.csv(gen$certificates, file.path(OUT, "certificates_guard.csv"), row.names = FALSE)
  jsonlite::write_json(list(formulation_id = E20$base_formulation, kind = "mga", variant = ARM$label, floor_g = FLOOR_G, blocks = BLOCKS_ARM,
                            band_g = as.numeric(E20$band_g), k = gen$k, anchor_objective_resolved = anchor$z,
                            total_runtime_s = sum(gen$certificates$runtime_s), wall_s = proc.time()[["elapsed"]] - t0,
                            created_utc = format(Sys.time(), tz = "UTC")),
                       file.path(OUT, "guard_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
  cat(sprintf("E20 guarded run complete: anchor %.6f | %d members | %.0f s solver / %.0f s wall -> %s\n",
              anchor$z, gen$k, sum(gen$certificates$runtime_s), proc.time()[["elapsed"]] - t0, OUT_REL))
}
cat("both arms done -- next: 18g_e20_analysis\n")


== e20_consistent: PF-2: consistent set -- saturating refugia + convex transboundary connectivity (weight only, t = 1) + convex corridors
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 28 features (8 continuous + 20 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 28 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget
   climate_type_macrorefugia: test layer confirmed (max/mean 7.31)
   transboundary_connectivity: test layer confirmed (max/mean 1052.96)
   climate_corridors: test layer confirmed (max/mean 8.86)
   transboundary target

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0.05 and 1.880196)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 4.848516 (bound 4.848481, gap 7.13e-06) | 381,874 selected | 235 s
band wall appended: obj0 . x <= 5.090941  (g = 0.05 on z* = 4.848516)
  block floor core_habitat   anchor capture 0.4315 -> floor 0.4100
  block floor connectivity   anchor capture 0.7405 -> floor 0.7035
  block floor carbon         anchor capture 0.6892 -> floor 0.6548
  block floor biodiversity   anchor capture 0.6539 -> floor 0.6212
g=0.05 iter 01/50: band 5.090813 (+5.00% of z*) OK | ham(anchor) 287,712 | 62 s
g=0.05 iter 02/50: band 5.090825 (+5.00% of z*) OK | ham(anchor) 201,124 | 64 s
g=0.05 iter 03/50: band 5.090939 (+5.00% of z*) OK | ham(anchor) 180,984 | 66 s
g=0.05 iter 04/50: band 5.090920 (+5.00% of z*) OK | ham(anchor) 188,614 | 61 s
g=0.05 iter 05/50: band 5.090940 (+5.00% of z*) OK | ham(anchor) 185,456 | 63 s
g=0.05 iter 06/50: band 5.090892 (+5.00% of z*) OK | ham(anchor) 183,112 | 62 s
g=0.05

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0.05 and 1.899525)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 4.627386 (bound 4.627315, gap 1.56e-05) | 381,874 selected | 260 s
band wall appended: obj0 . x <= 4.858756  (g = 0.05 on z* = 4.627386)
  block floor core_habitat   anchor capture 0.4424 -> floor 0.4202
  block floor connectivity   anchor capture 0.5919 -> floor 0.5623
  block floor carbon         anchor capture 0.6986 -> floor 0.6637
  block floor biodiversity   anchor capture 0.6558 -> floor 0.6230
g=0.05 iter 01/50: band 4.858728 (+5.00% of z*) OK | ham(anchor) 282,804 | 55 s
g=0.05 iter 02/50: band 4.858754 (+5.00% of z*) OK | ham(anchor) 195,300 | 58 s
g=0.05 iter 03/50: band 4.854437 (+4.91% of z*) OK | ham(anchor) 182,164 | 54 s
g=0.05 iter 04/50: band 4.808674 (+3.92% of z*) OK | ham(anchor) 178,930 | 57 s
g=0.05 iter 05/50: band 4.810690 (+3.96% of z*) OK | ham(anchor) 178,218 | 57 s
g=0.05 iter 06/50: band 4.808683 (+3.92% of z*) OK | ham(anchor) 177,426 | 57 s
g=0.05

...

compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 4.872439 (bound 4.872373, gap 1.36e-05) | 381,874 selected | 138 s
band wall appended: obj0 . x <= 5.116061  (g = 0.05 on z* = 4.872439)
  block floor core_habitat   anchor capture 0.3847 -> floor 0.3654
  block floor transboundary  anchor capture 0.5820 -> floor 0.5529
  block floor corridors      anchor capture 0.3293 -> floor 0.3128
  block floor carbon         anchor capture 0.6395 -> floor 0.6075
  block floor biodiversity   anchor capture 0.6064 -> floor 0.5760
g=0.05 iter 01/50: band 5.116051 (+5.00% of z*) OK | ham(anchor) 276,516 | 62 s
g=0.05 iter 02/50: band 5.116017 (+5.00% of z*) OK | ham(anchor) 179,964 | 61 s
g=0.05 iter 03/50: band 5.116059 (+5.00% of z*) OK | ham(anchor) 171,820 | 64 s
g=0.05 iter 04/50: band 5.116056 (+5.00% of z*) OK | ham(anchor) 170,464 | 64 s
g=0.05 iter 05/50: band 5.116030 (+5.00% of z*) OK | ham(anchor) 175,492 | 63 s
g=0.05 iter 06/50: 

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 1.680729)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 4.113775 (bound 4.113723, gap 1.27e-05) | 381,874 selected | 40 s
band wall appended: obj0 . x <= 4.319464  (g = 0.05 on z* = 4.113775)
  block floor core_habitat   anchor capture 0.4138 -> floor 0.3931
  block floor connectivity   anchor capture 0.5352 -> floor 0.5085
  block floor carbon         anchor capture 0.6608 -> floor 0.6278
  block floor biodiversity   anchor capture 0.6600 -> floor 0.6270
g=0.05 iter 01/50: band 4.319445 (+5.00% of z*) OK | ham(anchor) 272,372 | 50 s
g=0.05 iter 02/50: band 4.319463 (+5.00% of z*) OK | ham(anchor) 186,036 | 57 s
g=0.05 iter 03/50: band 4.319463 (+5.00% of z*) OK | ham(anchor) 168,268 | 55 s
g=0.05 iter 04/50: band 4.319464 (+5.00% of z*) OK | ham(anchor) 173,952 | 54 s
g=0.05 iter 05/50: band 4.319444 (+5.00% of z*) OK | ham(anchor) 172,878 | 55 s
g=0.05 iter 06/50: band 4.319465 (+5.00% of z*) OK | ham(anchor) 176,748 | 57 s
g=0.05 

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 1.897862)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 3.899922 (bound 3.899906, gap 4.00e-06) | 381,874 selected | 40 s
band wall appended: obj0 . x <= 4.094918  (g = 0.05 on z* = 3.899922)
  block floor core_habitat   anchor capture 0.3627 -> floor 0.3445
  block floor connectivity   anchor capture 0.6361 -> floor 0.6043
  block floor carbon         anchor capture 0.6111 -> floor 0.5805
  block floor biodiversity   anchor capture 0.6268 -> floor 0.5954
g=0.05 iter 01/50: band 4.094897 (+5.00% of z*) OK | ham(anchor) 223,510 | 61 s
g=0.05 iter 02/50: band 4.094902 (+5.00% of z*) OK | ham(anchor) 147,256 | 64 s
g=0.05 iter 03/50: band 4.094906 (+5.00% of z*) OK | ham(anchor) 141,734 | 59 s
g=0.05 iter 04/50: band 4.094918 (+5.00% of z*) OK | ham(anchor) 138,180 | 60 s
g=0.05 iter 05/50: band 4.094918 (+5.00% of z*) OK | ham(anchor) 142,962 | 55 s
g=0.05 iter 06/50: band 4.094895 (+5.00% of z*) OK | ham(anchor) 140,142 | 53 s
g=0.05 